# Reliability and Sensitivity Analysis

Research notebook for examining scoring reliability, confidence intervals, sensitivity and leave-one-criterion-out robustness in heritage reconstruction evaluation.

> Input score/annotation files are intentionally not distributed in this public portfolio version.


In [ ]:
# ============================================================
# Q1 RESULTS PIPELINE — RELIABILITY • SENSITIVITY • CIs • LOCO
# ============================================================
import os, json, hashlib, platform, sys, glob, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(7)

# -----------------------------
# Paths / output directories
# -----------------------------
os.makedirs("./out_fcs", exist_ok=True)
os.makedirs("./out", exist_ok=True)
os.makedirs("./out/figures", exist_ok=True)

CAL_FILE = "./out_fcs/fcs_scores_base_calibrated.csv"
ANN1 = "./annotations/ann1.csv"
ANN2 = "./annotations/ann2.csv"

# -----------------------------
# Helper: save weights
# -----------------------------
def save_weights(weights: dict):
    with open("./out_fcs/feature_weights.json","w", encoding="utf-8") as f:
        json.dump(weights, f, indent=2)

# -----------------------------
# 0) Load or build the core score table (df)
#    Priority: two annotators -> consensus -> calibrated [0,1]
#    Fallback: existing calibrated CSV
# -----------------------------
df = None
if os.path.exists(ANN1) and os.path.exists(ANN2):
    a1 = pd.read_csv(ANN1).set_index("image")
    a2 = pd.read_csv(ANN2).set_index("image")
    CRITERIA = [c for c in a1.columns if c in a2.columns]
    # detect annotator scale (0..K)
    max_scale = float(max(a1[CRITERIA].to_numpy().max(), a2[CRITERIA].to_numpy().max()))
    if max_scale <= 0: max_scale = 1.0
    consensus = (a1[CRITERIA] + a2[CRITERIA]) / 2.0
    df = (consensus / max_scale).clip(0,1).reset_index()   # normalized to [0,1]
    df.to_csv(CAL_FILE, index=False)
    print(f"Built consensus from annotators and saved -> {CAL_FILE}")
elif os.path.exists(CAL_FILE):
    df = pd.read_csv(CAL_FILE)
    print(f"Loaded calibrated scores -> {CAL_FILE}")
else:
    raise RuntimeError("No data to evaluate. Provide annotator CSVs OR a calibrated score table in ./out_fcs/.")

assert "image" in df.columns, "The table must have an 'image' column."
images = df["image"].tolist()

# -----------------------------
# 1) Feature weights
#    If your notebook defined FEATURE_WEIGHTS already, we will keep it.
#    Otherwise, equal weights are used (edit here if needed).
# -----------------------------
try:
    FEATURE_WEIGHTS  # from your earlier cells
    WEIGHTS = FEATURE_WEIGHTS.copy()
    print("Using FEATURE_WEIGHTS from the notebook.")
except NameError:
    crit_cols = [c for c in df.columns if c!="image" and not c.endswith("_conf") and c!="aggregate_weighted"]
    WEIGHTS = {c: 1.0/len(crit_cols) for c in crit_cols}
    print("No FEATURE_WEIGHTS in memory. Using equal weights.")
save_weights(WEIGHTS)

# Canonical criteria set
CRITERIA = [c for c in df.columns if c!="image" and not c.endswith("_conf") and c!="aggregate_weighted"]

# -----------------------------
# 2) Drop non-informative criteria (variance≈0); renormalize weights
# -----------------------------
var = df[CRITERIA].var()
INF = [c for c in CRITERIA if var[c] >= 1e-6]
if len(INF) < 2:
    print("WARNING: <2 informative criteria. Add annotator scores or improve automated scoring to create spread.")
CRITERIA = INF
WEIGHTS = {c: WEIGHTS[c] for c in CRITERIA}
s = sum(WEIGHTS.values()); WEIGHTS = {k: v/s for k,v in WEIGHTS.items()}
save_weights(WEIGHTS)
print("Informative criteria:", CRITERIA)

# -----------------------------
# 3) Baseline aggregate ranking
# -----------------------------
df_base = df.set_index("image")
agg = df_base[CRITERIA].dot(pd.Series(WEIGHTS))
ranked = agg.sort_values(ascending=False)
ranked_df = ranked.reset_index().rename(columns={0:"aggregate_weighted"})
ranked_df.columns = ["image","aggregate_weighted"]
ranked_df.to_csv("./out/overall_ranking.csv", index=False)
print("Saved baseline ranking -> ./out/overall_ranking.csv")

plt.figure()
plt.barh(ranked_df["image"], ranked_df["aggregate_weighted"])
plt.xlabel("Aggregate score (weighted)")
plt.ylabel("Image")
plt.title("Overall ranking (baseline)")
plt.tight_layout()
plt.savefig("./out/figures/Fig_overall_ranking.png", dpi=300); plt.show()

# =================================================================
# A) Inter-rater reliability (if annotator files exist): κ + ICC(2,k)
# =================================================================
def kappa(x, y):
    x, y = np.asarray(x), np.asarray(y)
    labs = sorted(set(x.tolist()+y.tolist()))
    po = np.mean(x==y)
    pa = np.array([np.mean(x==l) for l in labs])
    pb = np.array([np.mean(y==l) for l in labs])
    pe = float((pa*pb).sum())
    return 1.0 if pe==1 else (po-pe)/(1-pe)

def icc2k(M):
    X = np.asarray(M, float); n,k = X.shape
    mr = X.mean(0); ms = X.mean(1); g = X.mean()
    MSR = k*np.sum((ms-g)**2)/(n-1)
    MSC = n*np.sum((mr-g)**2)/(k-1) if k>1 else 0.0
    MSE = np.sum((X-ms[:,None]-mr[None,:]+g)**2)/((n-1)*max(k-1,1)) if k>1 else 0.0
    return float((MSR - MSE) / (MSR + (MSC - MSE)/n))

if os.path.exists(ANN1) and os.path.exists(ANN2):
    a1 = pd.read_csv(ANN1).set_index("image").loc[images]
    a2 = pd.read_csv(ANN2).set_index("image").loc[images]
    crits = [c for c in CRITERIA if c in a1.columns and c in a2.columns]
    rows=[]
    for c in crits:
        rows.append({"criterion":c, "kappa": kappa(a1[c], a2[c])})
    df_k = pd.DataFrame(rows).sort_values("criterion")
    df_k.to_csv("./out/reliability_kappa.csv", index=False)
    r1 = a1[crits].mean(1).values
    r2 = a2[crits].mean(1).values
    ICC = icc2k(np.vstack([r1,r2]).T)
    json.dump({"ICC2k": ICC}, open("./out/reliability_icc.json","w"), indent=2)

    plt.figure()
    plt.barh(df_k["criterion"], df_k["kappa"])
    plt.xlabel("Cohen's κ"); plt.ylabel("Criterion")
    plt.title("Inter-rater reliability")
    plt.tight_layout()
    plt.savefig("./out/figures/Fig_R1a_kappa.png", dpi=300); plt.show()
    print("Reliability saved to ./out")
else:
    print("Reliability skipped (provide ./annotations/ann1.csv and ann2.csv).")

# ============================================
# B) Weight sensitivity (Kendall’s τ, Top-1/3)
# ============================================
def rank_with(w):
    s=sum(w.values()); w={k:v/s for k,v in w.items()}
    return df_base[CRITERIA].dot(pd.Series(w)).sort_values(ascending=False)

def kendall_tau(a: pd.Series, b: pd.Series):
    ia = {k:i for i,k in enumerate(a.index)}; ib = {k:i for i,k in enumerate(b.index)}
    keys=list(ia.keys()); n=len(keys); C=D=0
    for i in range(n):
        for j in range(i+1,n):
            k1,k2 = keys[i], keys[j]
            s = (ia[k1]-ia[k2])*(ib[k1]-ib[k2])
            if   s>0: C+=1
            elif s<0: D+=1
    return (C-D)/max(C+D,1)

base = rank_with(WEIGHTS)
top2 = [k for k,_ in sorted(WEIGHTS.items(), key=lambda kv:-kv[1])[:2]]
rows=[]
for pct in [0.05,0.10,0.20]:
    for sgn in (+1,-1):
        Wp = WEIGHTS.copy()
        for k in top2: Wp[k] = max(0.0, WEIGHTS[k]*(1+sgn*pct))
        r = rank_with(Wp)
        rows.append({"pct": f"{'+' if sgn>0 else '-'}{int(pct*100)}%",
                     "tau": float(kendall_tau(base, r)),
                     "top1_same": int(r.index[0]==base.index[0]),
                     "top3_same": int(set(r.index[:3])==set(base.index[:3]))})
sens = pd.DataFrame(rows)
sens.to_csv("./out/sensitivity_weights.csv", index=False)

plt.figure()
g = sens.groupby("pct")["tau"].mean().reset_index()
plt.bar(g["pct"], g["tau"])
plt.xlabel("Weight change on two largest weights"); plt.ylabel("Kendall's τ")
plt.title("Ranking sensitivity to weight shifts")
plt.tight_layout(); plt.savefig("./out/figures/Fig_R2a_tau.png", dpi=300); plt.show()

plt.figure()
g2 = sens.groupby("pct")["top1_same"].mean().reindex(["+5%","-5%","+10%","-10%","+20%","-20%"])
plt.bar(g2.index, g2.values); plt.ylim(0,1)
plt.xlabel("Perturbation"); plt.ylabel("Top-1 stability")
plt.title("Top-1 stability under weight shifts")
plt.tight_layout(); plt.savefig("./out/figures/Fig_R2b_top1.png", dpi=300); plt.show()

# ============================================
# C) Bootstrap CIs + rank probabilities
# ============================================
def mean_duplicate_columns(frame: pd.DataFrame) -> pd.DataFrame:
    if not frame.columns.duplicated().any():
        return frame
    return frame.T.groupby(level=0).mean().T

B=2000; noise=0.05
scores_boot = np.zeros((B, len(images)))
rank_counts = np.zeros((len(images), len(images)), dtype=int)

for b in range(B):
    crit = np.random.choice(CRITERIA, size=len(CRITERIA), replace=True)
    sub  = mean_duplicate_columns(df_base[crit])
    Wj   = {c: WEIGHTS[c]*(1+np.random.uniform(-noise,noise)) for c in sub.columns}
    S    = sum(Wj.values()); Wj = {k:v/S for k,v in Wj.items()}
    agg  = sub.dot(pd.Series(Wj))
    scores_boot[b,:] = agg.values
    order = np.argsort(-agg.values)
    for rnk, ix in enumerate(order): rank_counts[ix,rnk]+=1

mean = scores_boot.mean(0); lo = np.percentile(scores_boot,2.5,0); hi = np.percentile(scores_boot,97.5,0)
df_ci = pd.DataFrame({"image":images,"mean":mean,"ci_low":lo,"ci_high":hi}).sort_values("mean",ascending=False)
df_ci.to_csv("./out/bootstrap_ci.csv", index=False)

rows=[]
nB=float(B)
for i,img in enumerate(images):
    cum=np.cumsum(rank_counts[i,:])/nB
    for k in [1,3]:
        rows.append({"image":img,"k":k,"P_rank_le_k":float(cum[k-1])})
pd.DataFrame(rows).to_csv("./out/rank_probabilities.csv", index=False)

plt.figure()
order = df_ci["image"].tolist(); idx=[images.index(x) for x in order]
plt.errorbar(mean[idx], range(len(order)), xerr=[mean[idx]-lo[idx], hi[idx]-mean[idx]], fmt='o')
plt.yticks(range(len(order)), order)
plt.xlabel("Weighted score (bootstrap mean ±95% CI)"); plt.ylabel("Image")
plt.title("Bootstrap uncertainty (B=2000)")
plt.tight_layout(); plt.savefig("./out/figures/Fig_R3a_bootstrap_CI.png", dpi=300); plt.show()

P = rank_counts/nB
plt.figure()
plt.imshow(P, aspect='auto', interpolation='nearest')
plt.xlabel("Rank (0=best)"); plt.ylabel("Image index (CSV order)")
plt.title("Bootstrap rank probability")
plt.tight_layout(); plt.savefig("./out/figures/Fig_R3b_rank_prob.png", dpi=300); plt.show()

# ============================================
# D) LOCO — leave-one-criterion-out
# ============================================
def rank_list(crit):
    w = {c: WEIGHTS[c] for c in crit}; s=sum(w.values()); w={k:v/s for k,v in w.items()}
    return list(df_base[crit].dot(pd.Series(w)).sort_values(ascending=False).index)

base_top3 = rank_list(CRITERIA)[:3]
rows=[]
for c in CRITERIA:
    kept = [x for x in CRITERIA if x!=c]
    r = rank_list(kept)[:3]
    rows.append({"excluded":c, "top3_flipped": int(set(r)!=set(base_top3)),
                 "base_top3": ",".join(base_top3), "loco_top3": ",".join(r)})
pd.DataFrame(rows).to_csv("./out/loco.csv", index=False)

# ============================================
# E) Jitter robustness (optional; requires FEATURES/score_* earlier)
# ============================================
try:
    FEATURES  # name -> scoring function
    import cv2 as cv
    def load_img(p):
        im = cv.imread(p, cv.IMREAD_COLOR); 
        if im is None: raise FileNotFoundError(p)
        return im
    def jitter(im):
        h,w = im.shape[:2]; dx,dy = int(0.02*w), int(0.02*h)
        x1=np.random.randint(0,dx+1); y1=np.random.randint(0,dy+1)
        x2=w-np.random.randint(0,dx+1); y2=h-np.random.randint(0,dy+1)
        crop = cv.resize(im[y1:y2, x1:x2], (w,h))
        alpha = 1.0+np.random.uniform(-0.05,0.05); beta=np.random.uniform(-5,5)
        return cv.convertScaleAbs(crop, alpha=alpha, beta=beta)

    # collect candidate image paths (1.png..10.png)
    final_dir = "./final images"
    paths = sorted([p for p in glob.glob(os.path.join(final_dir,"*.png")) 
                    if os.path.basename(p).split(".")[0].isdigit()],
                   key=lambda p:int(os.path.basename(p).split(".")[0]))[:10]
    rows=[]
    for p in paths:
        im0 = load_img(p)
        for name, fn in FEATURES:
            vals=[fn(jitter(im0), p)[0] for _ in range(25)]
            rows.append({"image":os.path.basename(p),"criterion":name,
                         "mean":float(np.mean(vals)), "sd":float(np.std(vals,ddof=1))})
    dfj = pd.DataFrame(rows); dfj.to_csv("./out/jitter_robustness.csv", index=False)
    g = dfj.groupby("criterion")["sd"].mean().reset_index()
    plt.figure(); plt.bar(g["criterion"], g["sd"]); plt.xticks(rotation=90)
    plt.ylabel("Score SD under jitter"); plt.title("Framing/exposure robustness")
    plt.tight_layout(); plt.savefig("./out/figures/Fig_jitter_robustness.png", dpi=300); plt.show()
    print("Jitter robustness saved to ./out")
except Exception as e:
    print(f"Jitter robustness skipped: {e}")

# ============================================
# F) Edit-impact (optional; pre vs post scores)
# ============================================
PRE, POST = "./edits/pre_scores.csv", "./edits/post_scores.csv"
if os.path.exists(PRE) and os.path.exists(POST):
    pre=pd.read_csv(PRE).set_index("image"); post=pd.read_csv(POST).set_index("image")
    feats = [c for c in CRITERIA if c in pre.columns and c in post.columns]
    common = pre.index.intersection(post.index)
    rows=[]
    for f in feats:
        d=(post.loc[common,f]-pre.loc[common,f]).astype(float)
        m=d.mean(); s=d.std(ddof=1); d_eff=m/(s+1e-12); pr=float((d>0).mean())
        rows.append({"criterion":f,"delta_mean":float(m),"cohen_d":float(d_eff),"pass_rate":pr})
    pd.DataFrame(rows).to_csv("./out/edit_impact.csv", index=False)
    order=pd.read_csv("./out/edit_impact.csv").sort_values("cohen_d",ascending=False)
    plt.figure(); plt.bar(order["criterion"], order["cohen_d"]); plt.xticks(rotation=90)
    plt.ylabel("Cohen's d (paired Δ)"); plt.title("Edit-impact by criterion")
    plt.tight_layout(); plt.savefig("./out/figures/Fig_edit_impact.png", dpi=300); plt.show()
else:
    print("Edit-impact skipped (place ./edits/pre_scores.csv and ./edits/post_scores.csv).")

# ============================================
# G) Proxy evidence table (template CSV for manuscript Table E1)
# ============================================
proxy_rows=[{"class":"Geometry","claim":"Niche ~58×21×18 m; cave grid; white ground above head","strength":"Strong","notes":"Measured surveys"},
            {"class":"Iconography","claim":"Right Abhaya; left grasp drapery; three neck lines; five toes","strength":"Strong","notes":"Gandharan canons + fragments"},
            {"class":"Comparanda","claim":"Wavy S-waves, triangular hairline, compact bun (ushnisha)","strength":"Moderate","notes":"Scale extrapolation to 58 m"},
            {"class":"Pigments","claim":"Outer carmine; dark-blue crease ridges; undergarment unknown","strength":"Moderate","notes":"Conservative; avoid over-colorization"},
            {"class":"Archives","claim":"Drapery step folds; wooden cave closures; monastery low/forward","strength":"Moderate","notes":"Photographic and report concordance"}]
pd.DataFrame(proxy_rows).to_csv("./out/proxy_evidence_table.csv", index=False)

# ============================================
# H) Reproducibility manifest (hashes + env)
# ============================================
def sha256(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for chunk in iter(lambda:f.read(1<<20), b""):
            h.update(chunk)
    return h.hexdigest()

manifest={"python":sys.version,"platform":platform.platform(),
          "packages":{},"seeds":{"numpy":7},"files":[]}
for m in ("numpy","pandas","matplotlib","opencv-python","scipy"):
    try:
        mod=__import__(m.replace("-","_"))
        manifest["packages"][m]=getattr(mod,"__version__","unknown")
    except Exception: pass

for d in ["./final images","./final images/head","./final images/drapery","./out_fcs","./out","./annotations","./edits"]:
    if os.path.isdir(d):
        for p in glob.glob(os.path.join(d,"*")):
            if os.path.isfile(p):
                manifest["files"].append({"path":p,"sha256":sha256(p)})

json.dump(manifest, open("./out/manifest.json","w"), indent=2)
print("All outputs written to ./out and ./out/figures — manifest.json created.")


In [ ]:
import numpy as np
import cv2 as cv
import math

def _safe_norm01(x, lo, hi):
    if hi <= lo: return np.zeros_like(x, dtype=float) if hasattr(x,'__len__') else 0.0
    v = (x - lo) / (hi - lo)
    return float(np.clip(v, 0, 1))

def score_proportions(img, path=None):
    # continuous closeness to targets using a narrow Gaussian
    h, w = img.shape[:2]
    ed = cv.Canny(cv.cvtColor(img, cv.COLOR_BGR2GRAY), 80, 160)
    k  = cv.getStructuringElement(cv.MORPH_RECT,(5,5))
    edd= cv.morphologyEx(ed, cv.MORPH_CLOSE, k, iterations=2)
    cnts,_ = cv.findContours(edd, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    cnts = sorted(cnts, key=cv.contourArea, reverse=True)
    if not cnts: return 0.0, 0.3
    x,y,nw,nh = cv.boundingRect(cnts[0])
    statue = None
    for c in cnts[1:]:
        x2,y2,w2,h2 = cv.boundingRect(c)
        if w2*h2<0.004*w*h: continue
        if (x2>x and y2>y and x2+w2<x+nw and y2+h2<y+nh) and h2>0.3*h:
            statue=(x2,y2,w2,h2); break
    if statue is None: return 0.0, 0.4
    sx,sy,sw,sh = statue
    # ratios we can measure from 2D: shoulders/statue (use statue width), base/statue ~ niche width/statue height
    shoulders_to_statue = sw/max(sh,1e-6)
    base_to_statue      = nw/max(sh,1e-6)

    def closeness(v, target, tol):
        return math.exp(-((v-target)**2)/(2*tol*tol))

    s2 = closeness(shoulders_to_statue, 0.29, 0.06)
    s3 = closeness(base_to_statue,      0.36, 0.08)
    raw = 0.55*s2 + 0.45*s3
    return float(raw), 0.75

def score_head_hair(img, path=None):
    h,w = img.shape[:2]
    # crop typical head band
    x0,y0=int(0.25*w),int(0.08*h); x1,y1=int(0.75*w),int(0.35*h)
    g=cv.cvtColor(img[y0:y1,x0:x1], cv.COLOR_BGR2GRAY)
    if g.size==0: return 0.0,0.3
    ed=cv.Canny(g,60,120)
    blur=cv.GaussianBlur(g,(3,3),0).astype(np.float32)
    gy,gx=np.gradient(blur); ang=np.degrees(np.arctan2(gy,gx))%180
    sel=ed>0
    if sel.sum()<100: return 0.0,0.3
    # proportion of near-horizontal strokes (waves) minus circular curl penalty
    horiz = np.mean((ang[sel]>150) | (ang[sel]<30))
    circles=cv.HoughCircles(g,cv.HOUGH_GRADIENT,1.2,30,param1=80,param2=30,minRadius=6,maxRadius=60)
    penalty = 0.30 if circles is not None and len(circles[0])>5 else 0.0
    raw = max(0.0, horiz - penalty)              # continuous 0..1-ish
    return float(raw), min(0.9, 0.5+raw/2)

def score_neck_lines(img, path=None):
    h,w=img.shape[:2]
    roi=cv.cvtColor(img[int(0.30*h):int(0.42*h), int(0.35*w):int(0.65*w)], cv.COLOR_BGR2GRAY)
    if roi.size==0: return 0.0,0.3
    ed=cv.Canny(roi,50,120)
    lines=cv.HoughLinesP(ed,1,np.pi/180,30,minLineLength=20,maxLineGap=6)
    cnt=0
    if lines is not None:
        for x1,y1,x2,y2 in lines[:,0,:]:
            if abs(math.degrees(math.atan2(y2-y1,x2-x1)))<15: cnt+=1
    raw = min(1.0, cnt/3.0)                       # 0..1 by counting lines
    return float(raw), 0.6 if cnt else 0.5

def score_hands_pose(img, path=None):
    h,w=img.shape[:2]
    # Right palm vertical-ness (Abhaya)
    r = cv.cvtColor(img[int(0.38*h):int(0.60*h), int(0.45*w):int(0.62*w)], cv.COLOR_BGR2GRAY)
    e = cv.Canny(r,60,140)
    cnts,_=cv.findContours(e,cv.RETR_EXTERNAL,cv.CHAIN_APPROX_SIMPLE)
    ar_best=0
    for c in cnts:
        x,y,ww,hh=cv.boundingRect(c)
        if hh>0 and ww>0:
            ar = hh/ww
            if hh>0.15*r.shape[0] and 1.0<ar<5.0: ar_best=max(ar_best,ar)
    right = _safe_norm01(ar_best, 1.2, 3.5)       # higher aspect = more vertical palm

    # Left hand gripping drapery: edge density in left-lower torso box
    l = cv.cvtColor(img[int(0.50*h):int(0.72*h), int(0.30*w):int(0.48*w)], cv.COLOR_BGR2GRAY)
    dens = cv.Canny(l,60,140).mean()/255.0
    left = _safe_norm01(dens, 0.05, 0.20)
    raw = 0.6*right + 0.4*left
    return float(raw), 0.6+0.3*raw

def score_drapery(img, path=None):
    h,w=img.shape[:2]
    g=cv.cvtColor(img[int(0.35*h):int(0.75*h), :], cv.COLOR_BGR2GRAY)
    if g.size==0: return 0.0,0.3
    ed=cv.Canny(g,60,140)
    blur=cv.GaussianBlur(g,(5,5),0).astype(np.float32)
    gy,gx=np.gradient(blur); ang=(np.degrees(np.arctan2(gy,gx))%180)[ed>0]
    if ang.size<300: return 0.0,0.4
    # orientation regularity (low entropy), left-right asymmetry, central “cling”
    hist,_=np.histogram(ang,bins=18,range=(0,180)); p=hist/hist.sum()
    ent = -np.sum(p[p>0]*np.log(p[p>0]))/np.log(len(p))    # 0..1
    reg = 1.0 - ent                                       # regularity
    ww=ed.shape[1]; L = ed[:,:ww//2].mean()/255.0; R = ed[:,ww//2:].mean()/255.0
    asym = _safe_norm01(R-L, 0.00, 0.15)
    mid = ed[:, int(0.45*ww):int(0.55*ww)].mean()/255.0
    cling = _safe_norm01(mid, 0.05, 0.18)
    raw = 0.5*reg + 0.3*asym + 0.2*cling
    return float(raw), min(0.9,0.5+raw/2)

def score_undergarment(img, path=None):
    lab=cv.cvtColor(img,cv.COLOR_BGR2LAB); L,A,B=cv.split(lab)
    mask=((L>40)&(L<200)&(A>110)&(A<160)&(B>140)).astype(np.uint8)*255
    h,w=L.shape
    hem = mask[int(0.75*h):int(0.92*h), int(0.30*w):int(0.70*w)].mean()/255.0
    arm = mask[int(0.45*h):int(0.60*h), int(0.55*w):int(0.75*w)].mean()/255.0
    raw = (hem + arm)/2.0
    return float(raw), 0.6+0.3*raw

def score_robe_color(img, path=None):
    lab=cv.cvtColor(img,cv.COLOR_BGR2LAB); L,A,B=cv.split(lab)
    red  = (A>150).astype(np.uint8)*255
    blue = (B<120).astype(np.uint8)*255
    k=cv.getStructuringElement(cv.MORPH_RECT,(5,5))
    blue_d=cv.dilate(blue,k,1)
    adjacency=(cv.bitwise_and(red,blue_d).mean()/255.0)
    red_cov=red.mean()/255.0
    raw = 0.6*_safe_norm01(red_cov, 0.05, 0.35) + 0.4*_safe_norm01(adjacency, 0.02, 0.15)
    return float(raw), 0.5+0.35*raw

def score_caves_behind_feet(img, path=None):
    h,w=img.shape[:2]
    roi=img[int(0.80*h):int(0.98*h), int(0.30*w):int(0.70*w)]
    if roi.size==0: return 0.0,0.3
    gray=cv.cvtColor(roi,cv.COLOR_BGR2GRAY)
    th=cv.adaptiveThreshold(cv.GaussianBlur(gray,(5,5),0),255,
                            cv.ADAPTIVE_THRESH_MEAN_C,cv.THRESH_BINARY_INV,31,5)
    cnts,_=cv.findContours(th,cv.RETR_EXTERNAL,cv.CHAIN_APPROX_SIMPLE)
    score=0.0; total=0
    for c in cnts:
        x,y,ww,hh=cv.boundingRect(c); area=cv.contourArea(c)
        if area<50: continue
        ar = ww/max(hh,1e-6); hull=cv.convexHull(c); sol=area/max(cv.contourArea(hull),1e-6)
        total += 1
        if 0.6<=ar<=2.2 and 0.5<=sol<=0.95: score += 1
    raw = 0.0 if total==0 else score/total
    return float(raw), 0.5+0.3*raw

def score_wooden_gates(img, path=None):
    h,w=img.shape[:2]
    roi=cv.cvtColor(img[int(0.70*h):int(0.98*h), int(0.10*w):int(0.90*w)], cv.COLOR_BGR2GRAY)
    if roi.size==0: return 0.0,0.3
    ed=cv.Canny(roi,70,150)
    lines=cv.HoughLinesP(ed,1,np.pi/180,60,minLineLength=25,maxLineGap=5)
    if lines is None: return 0.0,0.4
    v=hcount=0
    for x1,y1,x2,y2 in lines[:,0,:]:
        ang=abs(math.degrees(math.atan2(y2-y1,x2-x1)))
        if ang>75: v+=1
        if ang<15: hcount+=1
    raw = min(1.0, (0.06*v + 0.03*hcount))         # continuous coverage estimate
    return float(raw), 0.5+0.3*raw

def score_niche_arch(img, path=None):
    h,w=img.shape[:2]
    roi=img[int(0.10*h):int(0.35*h), int(0.20*w):int(0.80*w)]
    if roi.size==0: return 0.0,0.3
    ed=cv.Canny(cv.cvtColor(roi,cv.COLOR_BGR2GRAY),60,140)
    lines=cv.HoughLines(ed,1,np.pi/180,120)
    line_count = 0 if lines is None else len(lines)
    L=cv.cvtColor(roi,cv.COLOR_BGR2LAB)[:,:,0]
    band=L[int(0.10*L.shape[0]):int(0.25*L.shape[0]), :]
    bright = band.mean()
    raw = 0.6*_safe_norm01(line_count, 8, 25) + 0.4*_safe_norm01(bright, 150, 200)
    return float(raw), 0.55+0.3*_safe_norm01(line_count, 8, 25)


In [ ]:
FEATURE_FOLDER_ALIASES = {
    "head_hair": ["head_hair", "head", "hair"],
    "drapery": ["drapery"],
    # ...
}


In [ ]:
import os, glob
print("HEAD pool exists?:", os.path.isdir("./final images/head"))
print("HEAD files:", len(glob.glob("./final images/head/*")))


In [ ]:
# === SCORE base set (raw) ===
def load_img(path):
    im = cv.imread(path, cv.IMREAD_COLOR)
    if im is None:
        raise FileNotFoundError(path)
    return im

rows = []
for p in base_paths:
    img = load_img(p)
    rec = {"image": os.path.basename(p)}
    for feat_name, fn in FEATURES:
        sc, conf = fn(img, p)
        rec[feat_name] = float(sc)
        rec[feat_name + "_conf"] = float(conf)
    rows.append(rec)

df_raw = pd.DataFrame(rows)
df_raw.to_csv(os.path.join(OUT_DIR, "fcs_scores_base_raw.csv"), index=False)
print("Saved raw scores:", os.path.join(OUT_DIR, "fcs_scores_base_raw.csv"))

# list feature columns
feature_cols = [c for c in df_raw.columns if c not in ("image","aggregate_weighted") and not c.endswith("_conf")]
print("Features:", feature_cols)


In [ ]:
# === CALIBRATION & OVERALL RANKING ===
FEATURE_WEIGHTS = {
    "proportions"      : 0.20,
    "head_hair"        : 0.15,
    "neck_lines"       : 0.07,
    "hands_pose"       : 0.15,
    "drapery"          : 0.18,
    "undergarment"     : 0.06,
    "robe_color"       : 0.07,
    "caves_behind_feet": 0.06,
    "wooden_gates"     : 0.03,
    "niche_arch"       : 0.03,
}

cal_df = df_raw.copy()
for c in feature_cols:
    v = cal_df[c].values.astype(float)
    lo, hi = np.quantile(v, 0.05), np.quantile(v, 0.95)
    if hi <= lo:
        cal = v
    else:
        cal = (v - lo) / (hi - lo)
        cal = np.clip(cal, 0, 1)
    cal_df[c] = cal

def aggregate_weighted(row, weights):
    return float(sum(weights[k]*row.get(k,0.0) for k in weights.keys()))

cal_df["aggregate_weighted"] = cal_df.apply(lambda r: aggregate_weighted(r, FEATURE_WEIGHTS), axis=1)
cal_df.to_csv(os.path.join(OUT_DIR, "fcs_scores_base_calibrated.csv"), index=False)

df_rank = cal_df[["image","aggregate_weighted"]].sort_values("aggregate_weighted", ascending=False)
df_rank.to_csv(os.path.join(OUT_DIR, "overall_ranking.csv"), index=False)

print("Saved calibrated scores:", os.path.join(OUT_DIR, "fcs_scores_base_calibrated.csv"))
print("Saved overall ranking:", os.path.join(OUT_DIR, "overall_ranking.csv"))
df_rank


In [ ]:
# === PER-FEATURE POOLS: evaluate ALL files and pick Top-K for reporting ===
def score_pool_all(feature_name, score_fn):
    folder = None
    for alias in FEATURE_FOLDER_ALIASES.get(feature_name, [feature_name]):
        d = os.path.join(FEATURE_POOLS_DIR, alias)
        if os.path.isdir(d):
            folder = d; break
    if folder is None:
        return None, None  # no pool

    paths = sorted([p for p in glob.glob(os.path.join(folder, "*"))
                   if os.path.splitext(p)[1].lower() in (".png",".jpg",".jpeg",".tif",".tiff",".webp")])
    if not paths:
        return None, None

    recs = []
    for p in paths:
        img = load_img(p)
        sc, conf = name_to_fn[feature_name](img, p)
        recs.append({"image": os.path.basename(p), "score": float(sc), "conf": float(conf)})

    dfp = pd.DataFrame(recs).sort_values(["score","conf"], ascending=False)
    winners_for_report = dfp.head(TOP_K)
    winners_out = [{"image": r["image"], "score": float(r["score"]), "conf": float(r["conf"])}
                   for _, r in winners_for_report.iterrows()]
    return winners_out, dfp

# Start with base winners (so each feature has a result even without a pool)
best_by_feature = {}
for name in feature_cols:
    sort_cols = [name, f"{name}_conf"]
    if "aggregate_weighted" in cal_df.columns:
        sort_cols.append("aggregate_weighted")
    top = cal_df.sort_values(sort_cols, ascending=False).head(TOP_K)
    best_by_feature[name] = [{"image": r["image"], "score": float(r[name]), "conf": float(r.get(f"{name}_conf", 0.0))}
                             for _, r in top.iterrows()]

# Override with pools where they exist; also save full per-feature pool ranking
for feat_name, fn in FEATURES:
    winners, df_full = score_pool_all(feat_name, fn)
    if winners is not None:
        best_by_feature[feat_name] = winners
        df_full.to_csv(os.path.join(OUT_DIR, f"pool_ranking_{feat_name}.csv"), index=False)
        print(f"Pool for '{feat_name}': {len(df_full)} files -> top {TOP_K}: {[w['image'] for w in winners]}")
    else:
        print(f"No pool for '{feat_name}'")

with open(os.path.join(OUT_DIR, "best_by_feature.json"), "w", encoding="utf-8") as f:
    json.dump(best_by_feature, f, indent=2)

print("Saved:", os.path.join(OUT_DIR, "best_by_feature.json"))


In [ ]:
# === VISUALS ===
import matplotlib.pyplot as plt

# Overall ranking (calibrated)
disp = df_rank.copy()
plt.figure()
# horizontal bars (lowest at bottom)
plt.barh(disp["image"], disp["aggregate_weighted"])
plt.xlabel("Aggregate score (weighted, calibrated)")
plt.ylabel("Image")
plt.title("Overall Ranking (10 base images)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "fig_overall_ranking.png"), dpi=300)
plt.show()

# Feature heatmap (calibrated)
hm = cal_df.set_index("image")[feature_cols].clip(0,1)
plt.figure()
plt.imshow(hm.values, aspect="auto")
plt.xticks(ticks=np.arange(len(feature_cols)), labels=feature_cols, rotation=90)
plt.yticks(ticks=np.arange(hm.shape[0]), labels=hm.index.tolist())
plt.title("Feature Scores Heatmap (calibrated, 10 base images)")
plt.colorbar()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "fig_feature_heatmap.png"), dpi=300)
plt.show()

print("Saved charts to:", OUT_DIR)
